<a href="https://colab.research.google.com/github/ketanp23/LLMclass/blob/main/Lab0_Ngram_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 0 · N-gram Language Models (from scratch)

**Course:** Language Models — From N-grams to Transformers
**Goal:** Understand the oldest and simplest language model by building one in pure Python.

By the end of this lab you will be able to:
- Tokenize text and count **n-grams**
- Turn counts into next-word **probabilities** (with smoothing)
- **Generate** new text from the model
- Measure quality with **perplexity**
- See *first-hand* why n-grams break down on long-range context

> This notebook uses only the Python standard library — nothing to install. Just press **▶ Run** on each cell.


## 1 · A tiny corpus

A language model learns from text. We'll use a small, repetitive corpus so the patterns are easy to see. In practice you'd train on billions of words.

In [1]:
corpus = """
the cat sat on the mat .
the dog sat on the log .
the cat chased the dog .
the dog chased the cat .
a cat sat on a mat .
a dog sat on a log .
the cat sat on the log .
the dog sat on the mat .
"""

# Tokenize: lowercase and split on whitespace
tokens = corpus.lower().split()
print(f"Total tokens: {len(tokens)}")
print(f"Vocabulary size: {len(set(tokens))}")
print("First 15 tokens:", tokens[:15])

Total tokens: 54
Vocabulary size: 10
First 15 tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat', '.', 'the', 'dog', 'sat', 'on', 'the', 'log', '.', 'the']


## 2 · Counting n-grams

An **n-gram** is a contiguous sequence of *n* tokens.
- **unigram**: `the`
- **bigram**: `the cat`
- **trigram**: `the cat sat`

The core idea (the *Markov assumption*) is to predict the next word using only the previous **n−1** words:

$$P(w_t \mid w_1, \dots, w_{t-1}) \approx P(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

In [2]:
from collections import defaultdict, Counter

def build_ngram_counts(tokens, n):
    """Return counts of (context) -> Counter(next_word)."""
    counts = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        context = tuple(tokens[i:i + n - 1])   # the first n-1 words
        target  = tokens[i + n - 1]            # the word to predict
        counts[context][target] += 1
    return counts

bigram_counts = build_ngram_counts(tokens, n=2)

# Look at what can follow the word "the"
print('After "the", we saw:')
for word, c in bigram_counts[("the",)].most_common():
    print(f"   {word:8s} {c} times")

After "the", we saw:
   cat      4 times
   dog      4 times
   mat      2 times
   log      2 times


## 3 · From counts to probabilities

We convert counts to probabilities by dividing by how often the context appeared. To avoid assigning **zero probability** to unseen combinations, we use **add-k (Laplace) smoothing**:

$$P(w \mid \text{context}) = \frac{\text{count}(\text{context}, w) + k}{\text{count}(\text{context}) + k \cdot |V|}$$

In [3]:
vocab = sorted(set(tokens))
V = len(vocab)

def prob(counts, context, word, k=0.1):
    context = tuple(context)
    ctx_counter = counts[context]
    ctx_total = sum(ctx_counter.values())
    return (ctx_counter[word] + k) / (ctx_total + k * V)

# Probability that "cat" follows "the"
p = prob(bigram_counts, ["the"], "cat")
print(f'P(cat | the)  = {p:.3f}')
print(f'P(dog | the)  = {prob(bigram_counts, ["the"], "dog"):.3f}')
print(f'P(mat | the)  = {prob(bigram_counts, ["the"], "mat"):.3f}   <- rarely follows "the"')

P(cat | the)  = 0.315
P(dog | the)  = 0.315
P(mat | the)  = 0.162   <- rarely follows "the"


## 4 · Generating text

To generate, we start from a context and repeatedly **sample** the next word from the model's probability distribution.

In [4]:
import random
random.seed(0)

def generate(counts, n, start, length=12):
    context = list(start)
    output = list(start)
    for _ in range(length):
        ctx = tuple(context[-(n - 1):])
        candidates = counts[ctx]
        if not candidates:              # unseen context -> stop
            break
        words = list(candidates.keys())
        weights = [prob(counts, ctx, w) for w in words]
        nxt = random.choices(words, weights=weights)[0]
        output.append(nxt)
        context.append(nxt)
        if nxt == ".":
            break
    return " ".join(output)

print("Bigram model samples:\n")
for _ in range(5):
    print("  ", generate(bigram_counts, n=2, start=["the"]))

Bigram model samples:

   the log .
   the mat .
   the dog sat on the mat .
   the log .
   the cat chased the cat .


### Try a trigram model

A trigram uses the previous **two** words as context. With more context the output looks more coherent — but the model also becomes far sparser (many 3-word contexts are never seen).

In [5]:
trigram_counts = build_ngram_counts(tokens, n=3)

print("Trigram model samples:\n")
for _ in range(5):
    print("  ", generate(trigram_counts, n=3, start=["the", "cat"]))

Trigram model samples:

   the cat .
   the cat .
   the cat .
   the cat sat on a log .
   the cat sat on the log .


## 5 · Measuring quality: perplexity

**Perplexity** measures how "surprised" the model is by real text. Lower is better — a perplexity of *P* means the model is, on average, as uncertain as if choosing uniformly among *P* words.

$$\text{PPL} = \exp\!\left(-\frac{1}{N}\sum_{t=1}^{N} \log P(w_t \mid \text{context})\right)$$

In [6]:
import math

def perplexity(counts, n, test_tokens, k=0.1):
    log_sum = 0.0
    N = 0
    for i in range(n - 1, len(test_tokens)):
        ctx = test_tokens[i - (n - 1):i]
        word = test_tokens[i]
        p = prob(counts, ctx, word, k)
        log_sum += math.log(p)
        N += 1
    return math.exp(-log_sum / N)

test = "the cat sat on the mat .".split()
print(f"Bigram  perplexity on test sentence: {perplexity(bigram_counts, 2, test):.2f}")
print(f"Trigram perplexity on test sentence: {perplexity(trigram_counts, 3, test):.2f}")

Bigram  perplexity on test sentence: 2.14
Trigram perplexity on test sentence: 1.78


## 6 · Where n-grams break down

Run the cell below. The model **cannot** connect "the animal" to "it" many words later — it only ever sees the previous 1–2 words. This *long-range dependency* problem is exactly what embeddings, RNNs, and eventually attention were invented to solve.

In [7]:
# A context the model has never seen -> it gives up immediately
unseen = ("purple", "elephant")
print(f'Contexts starting with {unseen}:', dict(trigram_counts[unseen]) or "NONE — model has no idea")

print("\nKey limitations of n-grams:")
for lim in [
    "No long-range context (only n-1 words back)",
    "Data sparsity: most word combinations are never observed",
    "Model size explodes as n grows",
    "No notion of similarity — 'cat' and 'kitten' are unrelated symbols",
]:
    print("  -", lim)

Contexts starting with ('purple', 'elephant'): NONE — model has no idea

Key limitations of n-grams:
  - No long-range context (only n-1 words back)
  - Data sparsity: most word combinations are never observed
  - Model size explodes as n grows
  - No notion of similarity — 'cat' and 'kitten' are unrelated symbols


## 7 · Your turn 🧪

1. **Bigger n:** Build a 4-gram model with `build_ngram_counts(tokens, n=4)`. How does generation change? What happens to the number of usable contexts?
2. **Smoothing:** Change `k` in `prob(...)` to `1.0` and to `0.001`. How does perplexity move?
3. **New corpus:** Replace `corpus` with a paragraph of your own text and re-run everything.
4. **Stretch:** Write a function that reports the fraction of test trigrams that were *never seen* in training (the sparsity problem, quantified).

In [8]:
# Scratch space for your experiments
